# 🚀 Qwen3.8-35B-A3B Sweet-Spot Benchmark & Server Launcher (2x Tesla T4)

### คำแนะนำก่อนเริ่ม:
1. ด้านขวา (Settings) -> **Accelerator**: เลือก **GPU T4 x 2**
2. ด้านขวา (Settings) -> **Internet**: ปรับเป็น **On**
3. *(ทางเลือก)* หากใช้ Cloudflare Named Tunnel (`api.zexnoz.dev`): ไปที่ **Add-ons** -> **Secrets** -> เพิ่ม `CF_TUNNEL_TOKEN` (หากไม่ใส่ ระบบจะเปิด Quick Tunnel ฟรีให้แทนโดยอัตโนมัติ)

### การใช้งาน:
- **Cell 1**: ค้นหา **Sweet-Spot Context Window (130k - 196k)** พร้อมทดสอบความเร็ว Decode / Prefill / Generation และ Memory (KV Cache `q8_0` + Flash Attention)
- **Cell 2**: รัน **Production Server (Context 196k / KV q8_0 / Batch 1024/512 / Flash Attention -fa on)** พร้อม Cloudflare Tunnel
- **Cell 3**: ทดสอบส่ง Prompt พูดคุยกับ Qwen3.8 ใน Notebook
- **Cell 4**: คำสั่งหยุดการทำงานทั้งหมด (Stop Services)

## 🧪 Cell 1: Sweet-Spot Finder & Performance Benchmark (Context 130k-196k, KV q8_0, -fa on)
ทดสอบว่าบน 2x Tesla T4 โมเดลนี้ดันถึง Context 130k-190k+ ได้หรือไม่ พร้อมวัดความเร็ว Decode (Prefill) และ Output Generation ที่ Batch 1024 / UBatch 512 ด้วย KV Cache `q8_0`

In [ ]:
# ==============================================================================
# Sweet-Spot Discovery Benchmark for Qwen3.8-35B-A3B (2x Tesla T4)
# ==============================================================================
import subprocess
import sys
from pathlib import Path

# ติดตั้ง dependencies เพิ่มเติม (ถ้าจำเป็น)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=False)

# รันสคริปต์ Benchmark ด้วย KV q8_0
try:
    import benchmark_sweet_spot
    benchmark_sweet_spot.main([
        "--context-sizes", "131072,147456,163840,180224,196608",
        "--batch-size", "1024",
        "--ubatch-size", "512",
        "--kv-cache", "q8_0"
    ])
except ImportError:
    print("Running direct benchmark runner...")
    import os
    os.system("python -m py_compile /kaggle/working/benchmark_sweet_spot.py 2>/dev/null")


## 🚀 Cell 2: Start 24/7 API Server (KV q8_0 + Flash Attention -fa on)
เริ่มรัน llama-server พร้อม Tensor Split 1:1, KV Cache q8_0, Flash Attention (-fa on), Context 196k และ Cloudflare Tunnel เพื่อเชื่อมต่อจากภายนอก

In [ ]:
# ==============================================================================
# Kaggle T4 x2 -> Qwen3.8-35B-A3B + Cloudflare Tunnel (api.zexnoz.dev)
# ==============================================================================
import json
import os
import re
import shutil
import subprocess
import sys
import tarfile
import time
import urllib.request
from pathlib import Path

# ================= Configuration =================
DEFAULT_MODEL_REPO = "empero-ai/Qwen3.8-35B-A3B-Distill-GGUF"
DEFAULT_MODEL_FILE = "Qwen3.8-35B-A3B-Q4_K_M.gguf"
DEFAULT_MODEL_URL = "https://huggingface.co/empero-ai/Qwen3.8-35B-A3B-Distill-GGUF/resolve/main/Qwen3.8-35B-A3B-Q4_K_M.gguf"
MODEL_ALIAS = "qwen3.8-35B-A3B"
STATIC_DOMAIN = "https://api.zexnoz.dev"
CONTEXT_SIZE = 196608        # 196k tokens (Sweet Spot)
BATCH_SIZE = 1024            # Batch size สำหรับ Context ลึก
UBATCH_SIZE = 512
KV_CACHE_TYPE = "q8_0"       # ปรับเป็น q8_0 สำหรับ Attention Precision สูงสุด
SERVER_PORT = 8080
API_KEY = "kilo-secret-key"
# =================================================

WORKDIR = Path("/kaggle/tmp/llm_server" if Path("/kaggle").exists() else "/tmp/llm_server")
BIN_DIR = WORKDIR / "bin"
MODEL_DIR = Path("/kaggle/tmp/models" if Path("/kaggle").exists() else "/tmp/models")
LOG_DIR = Path("/kaggle/working" if Path("/kaggle").exists() else WORKDIR / "logs")
CLOUDFLARED_BIN = WORKDIR / "cloudflared"
LLAMA_LOG = LOG_DIR / "llama-server.log"
CF_LOG = LOG_DIR / "cloudflared.log"
USER_AGENT = "kaggle-llm-installer/1.0"

def stop_services():
    print("🧹 Cleaning up existing processes...")
    os.system("pkill -9 -f '[l]lama-server' >/dev/null 2>&1")
    os.system("pkill -9 -f '[c]loudflared' >/dev/null 2>&1")
    time.sleep(1)
    print("✅ Cleanup complete.")

stop_services()

def run_cmd(cmd, check=True, capture=False, env=None):
    print(f"$ {' '.join(str(x) for x in cmd)}", flush=True)
    return subprocess.run(
        [str(x) for x in cmd],
        check=check,
        text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.STDOUT if capture else None,
        env=env,
    )

def setup_directories():
    WORKDIR.mkdir(parents=True, exist_ok=True)
    BIN_DIR.mkdir(parents=True, exist_ok=True)
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    LOG_DIR.mkdir(parents=True, exist_ok=True)

def check_gpu():
    if not shutil.which("nvidia-smi"):
        raise RuntimeError("nvidia-smi not found. Enable GPU accelerator.")
    out = run_cmd(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture=True).stdout
    print(f"GPUs detected:\n{out.strip()}")

def get_cf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("CF_TUNNEL_TOKEN")
    except Exception:
        return os.environ.get("CF_TUNNEL_TOKEN", "")

def install_prebuilt_llamacpp():
    server_bin = BIN_DIR / "llama-server"
    if server_bin.exists() and os.access(server_bin, os.X_OK):
        print(f"llama.cpp ready at {server_bin}")
        return server_bin

    found = [p for p in WORKDIR.rglob("llama-server") if p.is_file() and os.access(p, os.X_OK)]
    if found:
        shutil.copy2(found[0], server_bin)
        return server_bin

    print("Fetching prebuilt llama.cpp CUDA binaries (sm_75)...")
    download_url = None
    archive_name = "ubuntu-cuda-sm_75-x64.tar.xz"
    try:
        api_url = "https://api.github.com/repos/cloudlnkcn/llama.cpp/releases?per_page=5"
        req = urllib.request.Request(api_url, headers={"User-Agent": USER_AGENT})
        with urllib.request.urlopen(req, timeout=15) as resp:
            releases = json.loads(resp.read().decode())
            pattern = re.compile(r"ubuntu-cuda-sm_75-x64\.tar\.xz$", re.I)
            for rel in releases:
                for asset in rel.get("assets", []):
                    if pattern.search(asset["name"]):
                        download_url = asset["browser_download_url"]
                        archive_name = asset["name"]
                        break
                if download_url:
                    break
    except Exception as e:
        print(f"Warning checking github releases: {e}")

    if not download_url:
        download_url = "https://github.com/ai-dock/llama.cpp-cuda/releases/download/b9628/llama.cpp-b9628-cuda-12.8-amd64.tar.gz"
        archive_name = "llama.cpp-b9628-cuda-12.8-amd64.tar.gz"

    archive_path = WORKDIR / archive_name
    urllib.request.urlretrieve(download_url, archive_path)
    with tarfile.open(archive_path, "r:*") as tf:
        tf.extractall(WORKDIR)

    found = [p for p in WORKDIR.rglob("llama-server") if p.is_file()]
    if not found:
        raise RuntimeError("llama-server not found")
    real_server = found[0]
    real_server.chmod(real_server.stat().st_mode | 0o755)
    shutil.copy2(real_server, server_bin)
    return server_bin

def install_cloudflared():
    if CLOUDFLARED_BIN.exists() and os.access(CLOUDFLARED_BIN, os.X_OK):
        return CLOUDFLARED_BIN
    print("Downloading cloudflared...")
    url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    urllib.request.urlretrieve(url, CLOUDFLARED_BIN)
    CLOUDFLARED_BIN.chmod(0o755)
    return CLOUDFLARED_BIN

def download_model(repo_id, filename):
    if Path("/kaggle/input").exists():
        mounted = [p for p in Path("/kaggle/input").rglob("*.gguf") if "mmproj" not in p.name.lower()]
        for m in mounted:
            if filename.lower() in m.name.lower() or "qwen3.8" in m.name.lower():
                print(f"Mounted from Dataset: {m.name} ({m.stat().st_size / (1024**3):.2f} GB)")
                return m
        if mounted:
            print(f"Using dataset model: {mounted[0].name}")
            return mounted[0]

    target = MODEL_DIR / filename
    if target.exists() and target.stat().st_size > 1024**3:
        print(f"Using cached model: {target.name}")
        return target

    print(f"Downloading {filename} (~20.2 GB)...")
    url = DEFAULT_MODEL_URL
    if shutil.which("aria2c"):
        run_cmd(["aria2c", "-x", "16", "-s", "16", "-k", "1M", "-d", str(MODEL_DIR), "-o", filename, url])
    else:
        run_cmd(["wget", "--continue", "-O", str(target), url])
    return target

def start_server(server_bin, model_path, context_size):
    so_dirs = {str(p.parent.resolve()) for p in WORKDIR.rglob("*.so*") if p.is_file()}
    env = os.environ.copy()
    env["LD_LIBRARY_PATH"] = ":".join(so_dirs) + ":" + os.environ.get("LD_LIBRARY_PATH", "")
    env["CUDA_VISIBLE_DEVICES"] = "0,1"
    env["GGML_CUDA_NO_VMM"] = "1"

    help_text = subprocess.run([str(server_bin), "--help"], capture_output=True, text=True, env=env).stdout or ""

    cmd = [
        str(server_bin),
        "-m", str(model_path),
        "-ngl", "99",
        "-sm", "layer",
        "-c", str(context_size),
        "-b", str(BATCH_SIZE),
        "-ub", str(UBATCH_SIZE),
        "-np", "1",
        "--cache-type-k", KV_CACHE_TYPE,
        "--cache-type-v", KV_CACHE_TYPE,
        "--tensor-split", "1,1",
        "--host", "127.0.0.1",
        "--port", str(SERVER_PORT),
        "--alias", MODEL_ALIAS,
    ]
    # Always enable Flash Attention
    if "--flash-attn" in help_text:
        cmd.extend(["--flash-attn", "on"])
    else:
        cmd.extend(["-fa", "on"])
    if "-fit" in help_text or "--fit" in help_text:
        cmd.extend(["-fit", "off"])
    if API_KEY:
        cmd.extend(["--api-key", API_KEY])

    print(f"Starting llama-server (Context: {context_size:,}, Batch: {BATCH_SIZE}/{UBATCH_SIZE}, KV: {KV_CACHE_TYPE}, Flash Attention: ON)...")
    log_file = open(LLAMA_LOG, "w", encoding="utf-8", buffering=1)
    server_proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT, env=env, text=True)

    print("Waiting for llama-server to initialize in VRAM...")
    start_t = time.time()
    ready = False
    while time.time() - start_t < 360:
        if server_proc.poll() is not None:
            print("Server exited unexpectedly! Last log lines:")
            if LLAMA_LOG.exists():
                print("\n".join(LLAMA_LOG.read_text().splitlines()[-25:]))
            return
        try:
            req = urllib.request.Request(f"http://127.0.0.1:{SERVER_PORT}/v1/models", headers={"Authorization": f"Bearer {API_KEY}"})
            with urllib.request.urlopen(req, timeout=3) as resp:
                if resp.status == 200:
                    ready = True
                    break
        except Exception:
            pass
        time.sleep(2)

    if not ready:
        print("Server timed out waiting for readiness.")
        return

    print("Starting Cloudflare Tunnel...")
    cf_token = get_cf_token()
    cf_file = open(CF_LOG, "w", encoding="utf-8", buffering=1)
    public_url = None

    if cf_token:
        subprocess.Popen([str(CLOUDFLARED_BIN), "tunnel", "run", "--token", cf_token], stdout=cf_file, stderr=subprocess.STDOUT)
        public_url = STATIC_DOMAIN
    else:
        subprocess.Popen([str(CLOUDFLARED_BIN), "tunnel", "--url", f"http://127.0.0.1:{SERVER_PORT}"], stdout=cf_file, stderr=subprocess.STDOUT)
        start_cf = time.time()
        while time.time() - start_cf < 45:
            if CF_LOG.exists():
                match = re.search(r"https://[-0-9a-z]+\.trycloudflare\.com", CF_LOG.read_text(errors="replace"))
                if match:
                    public_url = match.group(0)
                    break
            time.sleep(2)

    print("\n" + "=" * 74)
    print("🚀 QWEN3.8-35B-A3B SERVER ONLINE (2x TESLA T4)")
    print("=" * 74)
    print(f"📡 Base URL       : {public_url}/v1" if public_url else f"🖥️ Local URL: http://127.0.0.1:{SERVER_PORT}/v1")
    print(f"🔑 API Key        : {API_KEY}")
    print(f"🤖 Model ID       : {MODEL_ALIAS}")
    print(f"🧠 Context Length : {context_size:,} tokens (196k)")
    print(f"⚡ Batch / UBatch : {BATCH_SIZE} / {UBATCH_SIZE} (KV: {KV_CACHE_TYPE} | Flash-Attn: ON)")
    print("=" * 74)
    print("\n✅ Server is live in background. You can run Cell 3 to test chatting immediately!")

setup_directories()
check_gpu()
server_bin = install_prebuilt_llamacpp()
install_cloudflared()
model_path = download_model(DEFAULT_MODEL_REPO, DEFAULT_MODEL_FILE)
start_server(server_bin, model_path, CONTEXT_SIZE)


## 💬 Cell 3: Test Chat with Qwen3.8-35B-A3B
ทดสอบส่งข้อความและรับคำตอบผ่าน OpenAI-compatible API ภายใน Notebook

In [ ]:
# ==============================================================================
# Test chat with Qwen3.8-35B-A3B
# ==============================================================================
import json
import urllib.request

def ask_qwen(prompt, max_tokens=512):
    url = 'http://127.0.0.1:8080/v1/chat/completions'
    payload = {
        'model': 'qwen3.8-35B-A3B',
        'messages': [{'role': 'user', 'content': prompt}],
        'max_tokens': max_tokens,
        'temperature': 0.7
    }
    req = urllib.request.Request(
        url,
        data=json.dumps(payload).encode('utf-8'),
        headers={
            'Content-Type': 'application/json',
            'Authorization': 'Bearer kilo-secret-key'
        }
    )
    with urllib.request.urlopen(req, timeout=120) as resp:
        res = json.loads(resp.read().decode('utf-8'))
        return res['choices'][0]['message']['content']

prompt = 'สวัสดีครับ ช่วยแนะนำตัวเองสั้นๆ บอกสเปกของโมเดล และอธิบายว่าทำไมสถาปัตยกรรม Active 3B ถึงทำงานได้รวดเร็ว'
print(f'👤 User: {prompt}\n')
print('🤖 Qwen 3.8 is thinking...')
reply = ask_qwen(prompt)
print(f'\n{reply}')


## 🛑 Cell 4: Stop all servers
คำสั่งหยุดการทำงานของเซิร์ฟเวอร์ทั้งหมดเพื่อคืน GPU Resources

In [ ]:
stop_services()
